# Categorical Encoding: From One-Hot to Embeddings

Most ML models require numeric inputs. Converting categorical features to numbers
is a fundamental skill with many tradeoffs.

| Method | Cardinality | Preserves Order | Leakage Risk | Dimensionality |
|--------|-------------|-----------------|--------------|----------------|
| One-hot | Low-Medium | No | None | High |
| Label/Ordinal | Any | Yes (if ordinal) | None | Low |
| Target encoding | Any | No | **High** | Low |
| Frequency encoding | Any | No | None | Low |
| Embeddings | Very high | Learned | None | Medium |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

## Create a Dataset with Categorical Features

In [ ]:
n = 5000

# Low cardinality
colors = np.random.choice(['red', 'blue', 'green', 'yellow'], n)
sizes = np.random.choice(['S', 'M', 'L', 'XL'], n, p=[0.2, 0.35, 0.3, 0.15])

# High cardinality
n_cities = 200
city_names = [f'city_{i:03d}' for i in range(n_cities)]
city_probs = np.random.dirichlet(np.ones(n_cities) * 0.5)  # Power-law-ish
cities = np.random.choice(city_names, n, p=city_probs)

# Numeric features
age = np.random.normal(35, 10, n).clip(18, 70)
income = np.random.lognormal(10.5, 0.5, n)

# Target depends on features
city_effect = {c: np.random.randn() * 0.5 for c in city_names}
color_effect = {'red': 0.5, 'blue': -0.3, 'green': 0.1, 'yellow': -0.2}
size_effect = {'S': -0.3, 'M': 0.0, 'L': 0.2, 'XL': 0.4}

logit = (0.02 * (age - 35) + 0.00001 * (income - 40000) +
         np.array([color_effect[c] for c in colors]) +
         np.array([size_effect[s] for s in sizes]) +
         np.array([city_effect[c] for c in cities]))
prob = 1 / (1 + np.exp(-logit))
target = (np.random.rand(n) < prob).astype(int)

df = pd.DataFrame({
    'color': colors, 'size': sizes, 'city': cities,
    'age': age, 'income': income, 'target': target
})

print(f"Shape: {df.shape}")
print(f"Unique cities: {df['city'].nunique()}")
print(f"Target balance: {df['target'].mean():.3f}")
df.head()

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

## 1. One-Hot Encoding

Create a binary column for each category. Simple and interpretable.

**Key implementation detail:** handle unseen categories at test time.

In [ ]:
class OneHotEncoder:
    """One-hot encoding with unseen category handling."""
    
    def __init__(self, drop_first=False):
        self.drop_first = drop_first
        self.categories_ = {}  # col -> sorted list of categories
    
    def fit(self, df, cols):
        self.cols = cols
        for col in cols:
            self.categories_[col] = sorted(df[col].unique())
        return self
    
    def transform(self, df):
        result = df.copy()
        for col in self.cols:
            cats = self.categories_[col]
            start = 1 if self.drop_first else 0
            for cat in cats[start:]:
                result[f'{col}_{cat}'] = (df[col] == cat).astype(int)
            result = result.drop(columns=[col])
        return result

ohe = OneHotEncoder()
ohe.fit(train_df, ['color', 'size'])
train_ohe = ohe.transform(train_df)
test_ohe = ohe.transform(test_df)

ohe_cols = [c for c in train_ohe.columns if 'color_' in c or 'size_' in c]
print(f"One-hot columns: {ohe_cols}")
print(f"Total features after OHE (no city): {len(train_ohe.columns) - 2}")  # -target, -city

# Problem with high cardinality
ohe_all = OneHotEncoder()
ohe_all.fit(train_df, ['color', 'size', 'city'])
train_ohe_all = ohe_all.transform(train_df)
print(f"Total features after OHE (with city): {len(train_ohe_all.columns) - 1}")
print("  -> High cardinality explodes dimensionality!")

## 2. Label / Ordinal Encoding

Map categories to integers. Only meaningful if there's a natural ordering.

**Warning:** applying label encoding to nominal features (no ordering) implies a false ordinal relationship.

In [ ]:
class OrdinalEncoder:
    """Ordinal encoding with optional custom ordering."""
    
    def __init__(self, orderings=None):
        self.orderings = orderings or {}  # col -> ordered list
        self.mappings_ = {}
    
    def fit(self, df, cols):
        self.cols = cols
        for col in cols:
            if col in self.orderings:
                cats = self.orderings[col]
            else:
                cats = sorted(df[col].unique())
            self.mappings_[col] = {cat: i for i, cat in enumerate(cats)}
        return self
    
    def transform(self, df):
        result = df.copy()
        for col in self.cols:
            default = -1  # unseen categories
            result[col] = df[col].map(self.mappings_[col]).fillna(default).astype(int)
        return result

# Size has natural ordering
ord_enc = OrdinalEncoder(orderings={'size': ['S', 'M', 'L', 'XL']})
ord_enc.fit(train_df, ['size'])
train_ord = ord_enc.transform(train_df)

print("Size mapping:", ord_enc.mappings_['size'])
print(train_ord[['size']].head(10).T)

## 3. Target Encoding (with Smoothing)

Replace each category with the mean of the target for that category.

$$\text{encoded}(c) = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}_{\text{global}}}{n_c + m}$$

where $n_c$ = count of category $c$, $m$ = smoothing parameter.

**Critical:** must be computed on training data only, with cross-validation or smoothing to prevent leakage.

In [ ]:
class TargetEncoder:
    """Target encoding with smoothing to prevent overfitting."""
    
    def __init__(self, smoothing=10):
        self.smoothing = smoothing
        self.encoding_ = {}
        self.global_mean_ = None
    
    def fit(self, df, cols, target_col):
        self.cols = cols
        self.global_mean_ = df[target_col].mean()
        
        for col in cols:
            stats = df.groupby(col)[target_col].agg(['mean', 'count'])
            # Bayesian smoothing: blend category mean with global mean
            smoothed = (stats['count'] * stats['mean'] + self.smoothing * self.global_mean_) / (
                stats['count'] + self.smoothing
            )
            self.encoding_[col] = smoothed.to_dict()
        return self
    
    def transform(self, df):
        result = df.copy()
        for col in self.cols:
            result[col] = df[col].map(self.encoding_[col]).fillna(self.global_mean_)
        return result
    
    def fit_transform_cv(self, df, cols, target_col, n_folds=5):
        """Leave-one-out style: encode using out-of-fold statistics to prevent leakage."""
        self.cols = cols
        self.global_mean_ = df[target_col].mean()
        result = df.copy()
        
        fold_indices = np.random.randint(0, n_folds, len(df))
        
        for col in cols:
            encoded_col = pd.Series(index=df.index, dtype=float)
            for fold in range(n_folds):
                train_mask = fold_indices != fold
                val_mask = fold_indices == fold
                
                stats = df.loc[train_mask].groupby(col)[target_col].agg(['mean', 'count'])
                smoothed = (stats['count'] * stats['mean'] + self.smoothing * self.global_mean_) / (
                    stats['count'] + self.smoothing
                )
                mapping = smoothed.to_dict()
                encoded_col.loc[val_mask] = df.loc[val_mask, col].map(mapping)
            
            encoded_col = encoded_col.fillna(self.global_mean_)
            result[col] = encoded_col
            
            # Store full-data encoding for test set
            stats = df.groupby(col)[target_col].agg(['mean', 'count'])
            smoothed = (stats['count'] * stats['mean'] + self.smoothing * self.global_mean_) / (
                stats['count'] + self.smoothing
            )
            self.encoding_[col] = smoothed.to_dict()
        
        return result

# Demo: naive vs CV target encoding
te_naive = TargetEncoder(smoothing=10)
te_naive.fit(train_df, ['city'], 'target')
train_te_naive = te_naive.transform(train_df)

te_cv = TargetEncoder(smoothing=10)
train_te_cv = te_cv.fit_transform_cv(train_df, ['city'], 'target', n_folds=5)

print("Sample target encodings for 'city':")
sample_cities = list(te_naive.encoding_['city'].items())[:5]
for city, val in sample_cities:
    print(f"  {city}: {val:.4f}")
print(f"  Global mean: {te_naive.global_mean_:.4f}")

## 4. Frequency Encoding

Replace each category with its frequency (or normalized frequency) in the training data.
Simple, no target leakage, works well when frequency is predictive.

In [ ]:
class FrequencyEncoder:
    """Frequency (count) encoding."""
    
    def __init__(self, normalize=True):
        self.normalize = normalize
        self.freq_maps_ = {}
    
    def fit(self, df, cols):
        self.cols = cols
        for col in cols:
            counts = df[col].value_counts()
            if self.normalize:
                counts = counts / len(df)
            self.freq_maps_[col] = counts.to_dict()
        return self
    
    def transform(self, df):
        result = df.copy()
        for col in self.cols:
            default = 0 if not self.normalize else 1 / len(df)
            result[col] = df[col].map(self.freq_maps_[col]).fillna(default)
        return result

freq_enc = FrequencyEncoder(normalize=True)
freq_enc.fit(train_df, ['city', 'color'])
train_freq = freq_enc.transform(train_df)

print("Top 5 city frequencies:")
top5 = sorted(freq_enc.freq_maps_['city'].items(), key=lambda x: -x[1])[:5]
for city, freq in top5:
    print(f"  {city}: {freq:.4f}")

## 5. Entity Embeddings (Concept)

For very high cardinality features (user IDs, product IDs), learned embeddings map each category
to a dense vector in a low-dimensional space.

**How it works:**
- Each category gets an embedding vector (initially random)
- Embeddings are learned end-to-end during neural network training
- Similar categories end up close in embedding space

This is the same idea as word embeddings (Word2Vec) applied to tabular data.

In [ ]:
# Minimal embedding layer concept (no framework dependency)
class SimpleEmbedding:
    """Embedding lookup table with gradient descent updates."""
    
    def __init__(self, num_categories, embed_dim, lr=0.01):
        self.embed_dim = embed_dim
        self.lr = lr
        # Initialize embeddings randomly
        self.W = np.random.randn(num_categories, embed_dim) * 0.1
        self.cat_to_idx = {}
    
    def fit_mapping(self, categories):
        unique_cats = sorted(set(categories))
        self.cat_to_idx = {cat: i for i, cat in enumerate(unique_cats)}
    
    def lookup(self, categories):
        """Forward pass: look up embedding vectors."""
        indices = [self.cat_to_idx.get(c, 0) for c in categories]
        return self.W[indices]  # shape: (batch_size, embed_dim)
    
    def update(self, categories, gradients):
        """Update embeddings using gradients."""
        indices = [self.cat_to_idx.get(c, 0) for c in categories]
        for i, idx in enumerate(indices):
            self.W[idx] -= self.lr * gradients[i]

# Demo: show how embeddings are structured
emb = SimpleEmbedding(num_categories=200, embed_dim=8)
emb.fit_mapping(city_names)

sample = ['city_000', 'city_001', 'city_002']
vectors = emb.lookup(sample)

print("Embedding vectors (8-dim) for 3 cities:")
for city, vec in zip(sample, vectors):
    print(f"  {city}: [{', '.join(f'{v:.3f}' for v in vec)}]")

print(f"\nRule of thumb for embed_dim: min(50, num_categories // 2)")
print(f"  200 cities -> embed_dim ~ {min(50, 200 // 2)}")

## Comparison: Encoding Strategies on High-Cardinality Feature

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

numeric_cols = ['age', 'income']

def prepare_and_evaluate(train, test, name):
    """Prepare features and evaluate with a gradient boosted tree."""
    feature_cols = [c for c in train.columns if c != 'target']
    # Drop non-numeric columns if any remain
    feature_cols = [c for c in feature_cols if train[c].dtype in ['float64', 'int64', 'int32', 'float32']]
    
    X_train = train[feature_cols].values
    y_train = train['target'].values
    X_test = test[feature_cols].values
    y_test = test['target'].values
    
    model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    return acc, len(feature_cols)

results = {}

# 1. One-hot (all)
ohe_all = OneHotEncoder()
ohe_all.fit(train_df, ['color', 'size', 'city'])
tr_ohe = ohe_all.transform(train_df)
te_ohe = ohe_all.transform(test_df)
acc, nf = prepare_and_evaluate(tr_ohe, te_ohe, 'One-hot')
results['One-hot (all)'] = (acc, nf)

# 2. Label encoding
le = OrdinalEncoder(orderings={'size': ['S', 'M', 'L', 'XL']})
le.fit(train_df, ['color', 'size', 'city'])
tr_le = le.transform(train_df)
te_le = le.transform(test_df)
acc, nf = prepare_and_evaluate(tr_le, te_le, 'Label')
results['Label encoding'] = (acc, nf)

# 3. Target encoding (with CV)
te_enc = TargetEncoder(smoothing=10)
tr_te = te_enc.fit_transform_cv(train_df, ['color', 'size', 'city'], 'target')
te_te = te_enc.transform(test_df)
acc, nf = prepare_and_evaluate(tr_te, te_te, 'Target')
results['Target encoding'] = (acc, nf)

# 4. Frequency encoding
fe = FrequencyEncoder()
fe.fit(train_df, ['color', 'size', 'city'])
tr_fe = fe.transform(train_df)
te_fe = fe.transform(test_df)
acc, nf = prepare_and_evaluate(tr_fe, te_fe, 'Frequency')
results['Frequency encoding'] = (acc, nf)

# 5. One-hot (low card only) + target encoding (high card)
ohe_low = OneHotEncoder()
ohe_low.fit(train_df, ['color', 'size'])
tr_mix = ohe_low.transform(train_df)
te_mix = ohe_low.transform(test_df)

te_high = TargetEncoder(smoothing=10)
tr_mix = te_high.fit_transform_cv(tr_mix, ['city'], 'target')
te_mix = te_high.transform(te_mix)
acc, nf = prepare_and_evaluate(tr_mix, te_mix, 'Mixed')
results['OHE(low) + Target(high)'] = (acc, nf)

print(f"{'Method':<30} {'Accuracy':<10} {'# Features'}")
print("-" * 55)
for name, (acc, nf) in sorted(results.items(), key=lambda x: -x[1][0]):
    print(f"{name:<30} {acc:.4f}     {nf}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

names = list(results.keys())
accs = [results[n][0] for n in names]
nfeats = [results[n][1] for n in names]

bars = ax.barh(names, accs, color='steelblue')
for bar, acc, nf in zip(bars, accs, nfeats):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{acc:.4f} ({nf} features)', va='center', fontsize=9)

ax.set_xlabel('Accuracy')
ax.set_title('Encoding Strategy Comparison (GBT Classifier)')
ax.set_xlim(min(accs) - 0.02, max(accs) + 0.04)
plt.tight_layout()
plt.show()

## Notes: Hashing Trick

For extremely high cardinality (millions of categories), the **hashing trick** maps categories to a fixed number of buckets using a hash function:

```python
def hash_encode(value, n_buckets=1000):
    return hash(value) % n_buckets
```

**Pros:** Fixed memory, handles unseen categories, no fitting needed.  
**Cons:** Hash collisions (different categories map to same bucket), not interpretable.  
**Used in:** Vowpal Wabbit, sklearn's `HashingVectorizer`, online learning systems.

In [ ]:
def hash_encode(values, n_buckets=50):
    """Hash trick encoding."""
    result = np.zeros((len(values), n_buckets))
    for i, v in enumerate(values):
        bucket = hash(str(v)) % n_buckets
        result[i, bucket] = 1
    return result

hashed = hash_encode(train_df['city'].values, n_buckets=50)
print(f"Hashed shape: {hashed.shape} (200 cities -> 50 buckets)")
print(f"Collisions expected: ~{200 - 50 * (1 - ((50-1)/50)**200):.0f} categories share a bucket")

## Interview Questions

### Q: One-hot with high cardinality -- problems?
**A:** (1) Curse of dimensionality: creates thousands of sparse columns. (2) Most entries are zero, wasting memory and compute. (3) Many columns have very few 1s, leading to high variance / poor generalization. (4) Linear models struggle because each category gets its own independent weight with very little data.

**Better alternatives:** Target encoding (with smoothing/CV), frequency encoding, embeddings, or hashing trick.

### Q: Target encoding leakage -- how to prevent?
**A:** Target encoding uses the target variable, so naive application leaks information (the encoded value for each row partially contains its own label). Prevention:
1. **Leave-one-out / K-fold CV encoding:** compute encoding using only out-of-fold data
2. **Smoothing:** regularize toward global mean, especially for rare categories
3. **Always fit on training data only**, transform test separately

### Q: When to use embeddings vs one-hot?
**A:** Use **embeddings** when:
- Cardinality is high (>100 categories)
- Using a neural network model
- Categories have latent structure (similar cities, products, users)
- You want to capture interactions between category and other features

Use **one-hot** when:
- Cardinality is low (<20)
- Using tree-based models or linear models
- Interpretability matters